# Reglas de pre y post autorización

Paso 3 de la consigna. Las reglas usan el score por merchant (`risk_pct`, el percentil de riesgo) **y el volumen del merchant**, porque el score no ordena bien el fraude en general (ver `docs/decisions.md`, sección 4).

**Volumen:** intentos de transacción en los 30 días hasta el 30-sep, de la misma fila que se usó para el score. Cuatro niveles: sin actividad (sin dato), baja (1 a 9), media (10 a 99) y alta (100 o más).

**Qué se mide.** Las transacciones son de octubre a diciembre y nunca se usaron para entrenar. `is_fraud` y `approved` son el resultado:

- **Pre-autorización** (rechazar): fraude evitado, TPV bueno rechazado, tasa de aprobación resultante y *equilibrio* = fraude evitado / TPV bueno rechazado. Rechazar conviene si el margen que Clip gana sobre el TPV es menor que el equilibrio.
- **Post-autorización** (retener el pago, sobre lo aprobado): TPV retenido, fraude retenido, merchants a revisar y demora promedio.

**Supuestos que no están en los datos:** margen de Clip sobre el TPV = 1% y retención de 7 días. Hay que cambiarlos si se conocen los reales.

**Cómo se eligieron los cortes.** Con la tabla de cortes (abajo) y el criterio declarado de que una regla de rechazo debe tener un equilibrio por encima del margen supuesto con holgura (1,5% o más). La estructura se eligió mirando estas mismas transacciones: la separación en mitades de merchants muestra estabilidad, pero no es una evaluación independiente.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
MERCHANT_PATH = ROOT / "data" / "merchant_rating_featureset_sample.parquet"
SCORED_PATH = ROOT / "data" / "processed" / "transacciones_con_score.parquet"
SCORE_PATH = ROOT / "data" / "processed" / "score_octubre.parquet"
DOCS = ROOT / "docs" / "reglas"
DOCS.mkdir(parents=True, exist_ok=True)

MARGEN = 0.01       # supuesto: ganancia de Clip sobre el TPV
DIAS_RETENCION = 7  # supuesto: días que se retiene un pago
VOLUMENES = ["sin actividad", "baja", "media", "alta"]

t = pd.read_parquet(SCORED_PATH)
score = pd.read_parquet(SCORE_PATH, columns=["merchant_id", "observation_date"])

## Volumen de cada merchant

Se toma de la misma fila del score (1-oct, o la última anterior), para no usar información posterior al 30-sep.

In [ ]:
one = pd.Timestamp("2025-10-01")
cols = ["merchant_id", "observation_date", "me_attempted_txn_sum_30d"]
a = pd.read_parquet(MERCHANT_PATH, columns=cols, filters=[("observation_date", "=", one.date())])
other = score.loc[score["observation_date"] != one]
b = pd.read_parquet(MERCHANT_PATH, columns=cols, filters=[("merchant_id", "in", other["merchant_id"].tolist())])
for f in (a, b):
    f["observation_date"] = pd.to_datetime(f["observation_date"])
b = b.merge(other, on=["merchant_id", "observation_date"])
vol = pd.concat([a, b]).drop_duplicates("merchant_id")
vol = vol[vol["merchant_id"].isin(score["merchant_id"])]
intentos = vol["me_attempted_txn_sum_30d"].clip(lower=0)
vol["volumen"] = pd.cut(intentos.fillna(-1), [-2, -0.5, 9.5, 99.5, 1e12], labels=VOLUMENES).astype(str)
assert len(vol) == len(score)

t = t.merge(vol[["merchant_id", "volumen"]], on="merchant_id", how="left", validate="many_to_one")
assert t["volumen"].notna().all()
print(vol["volumen"].value_counts().reindex(VOLUMENES).to_string())

## Base: lo que pasa hoy sin reglas

In [ ]:
ap = (t["approved"] == 1).to_numpy()
fr = (t["is_fraud"] == 1).to_numpy()
amount = t["amount"].to_numpy()
risk = t["risk_pct"].to_numpy()
volumen = t["volumen"].to_numpy()
merchant = t["merchant_id"].to_numpy()

N = len(t)
APROBADAS = ap.sum()
TPV = amount[ap].sum()
TPV_BUENO = amount[ap & ~fr].sum()
FRAUDE_N = (ap & fr).sum()
FRAUDE_D = amount[ap & fr].sum()
print(f"{N:,} transacciones | aprobadas {APROBADAS:,} ({APROBADAS / N:.2%}) | TPV aprobado ${TPV / 1e9:.2f} mil millones")
print(f"fraude en lo aprobado: {FRAUDE_N:,} transacciones, ${FRAUDE_D / 1e6:.2f} millones ({FRAUDE_D / TPV:.3%} del TPV)")

In [ ]:
def pre_kpis(declined, label=""):
    d = declined & ap
    fd, gd = amount[d & fr].sum(), amount[d & ~fr].sum()
    return {
        "regla": label, "merchants": len(set(merchant[declined])), "transacciones rechazadas": int(d.sum()),
        "fraude evitado (txns)": int((d & fr).sum()), "fraude evitado (% del $)": fd / FRAUDE_D,
        "TPV bueno rechazado (% del bueno)": gd / TPV_BUENO, "equilibrio": fd / gd if gd else np.nan,
        "beneficio neto ($ millones)": (fd - MARGEN * gd) / 1e6,
        "aprobación final": (APROBADAS - d.sum()) / N,
    }


def post_kpis(held, base, label=""):
    h = held & base
    fd, gh = amount[h & fr].sum(), amount[h & ~fr].sum()
    good_total = amount[base & ~fr].sum()
    return {
        "regla": label, "merchants": len(set(merchant[h])), "TPV retenido (%)": amount[h].sum() / amount[base].sum(),
        "fraude retenido (txns)": int((h & fr).sum()), "fraude retenido (% del $)": fd / amount[base & fr].sum(),
        "demora promedio (días, TPV bueno)": DIAS_RETENCION * gh / good_total,
    }

## Tabla de cortes

Para cada nivel de volumen y cada corte de `risk_pct`: cuánto fraude se evita y cuánto TPV bueno se rechaza si se rechazan las transacciones de esos merchants.

In [ ]:
rows = []
for vols, name in ([["baja"], "baja"], [["media"], "media"], [["alta"], "alta"], [["baja", "media"], "baja + media"]):
    for cut in (0.90, 0.95, 0.99):
        m = (risk >= cut) & np.isin(volumen, vols)
        rows.append({"volumen": name, "corte": cut, **pre_kpis(m)})
cortes = pd.DataFrame(rows).drop(columns="regla")
cortes.round(4).to_csv(DOCS / "tabla_de_cortes.csv", index=False)
print(cortes[["volumen", "corte", "merchants", "fraude evitado (% del $)", "TPV bueno rechazado (% del bueno)", "equilibrio", "aprobación final"]].round(4).to_string(index=False))

## Reglas elegidas

**Pre-autorización (rechazar).** Solo donde el score discrimina y el equilibrio supera con holgura el margen:
- `risk_pct` de 0,90 o más y volumen **bajo**;
- `risk_pct` de 0,99 o más y volumen **medio**.

No se rechaza a los merchants de volumen **alto**, aunque tengan riesgo alto: ahí el score casi no separa el fraude (tasa de 0,05% a 0,07%) y rechazar costaría mucho TPV bueno.

**Post-autorización (retener, sobre lo aprobado y no rechazado):**
- **Revisión manual:** volumen **alto** con `risk_pct` de 0,95 o más. Pocos merchants y mucho TPV: no se rechaza, se retiene y se revisa.
- **Retención automática de 7 días:** merchants **sin actividad** en 30 días. El score los ubica en el riesgo más bajo, pero su tasa de fraude es 2,4 veces el promedio.
- El resto se paga de inmediato.

In [ ]:
PRE = ((risk >= 0.90) & (volumen == "baja")) | ((risk >= 0.99) & (volumen == "media"))
REVISION = (risk >= 0.95) & (volumen == "alta")
RETENCION = volumen == "sin actividad"

declined = PRE & ap
remaining = ap & ~declined
review = REVISION & remaining
auto = RETENCION & remaining & ~review

pre = pre_kpis(PRE, "Pre: rechazar")
rev = post_kpis(REVISION, remaining, "Post: revisión manual")
aut = post_kpis(RETENCION & ~REVISION, remaining, "Post: retención automática")
both = post_kpis(REVISION | RETENCION, remaining, "Post: ambas")

paso = ap & ~(review | auto) & ~declined
fd_total = amount[(declined | review | auto) & fr].sum()
print("PRE-AUTORIZACIÓN")
print(pd.Series(pre).drop("regla").round(4).to_string())
for k in (rev, aut, both):
    print()
    print(k["regla"])
    print(pd.Series(k).drop("regla").round(4).to_string())
print()
print(f"fraude que pasa sin ningún control: {int((paso & fr).sum()):,} de {FRAUDE_N:,} transacciones ({amount[paso & fr].sum() / FRAUDE_D:.1%} del $)")
print(f"fraude evitado o retenido: {fd_total / FRAUDE_D:.1%} del $")

In [ ]:
pd.DataFrame([pre, rev, aut, both]).round(5).to_csv(DOCS / "kpis_reglas.csv", index=False)

## ¿Se sostiene? Dos chequeos

1. **Mitades de merchants.** Se parten los merchants al azar en dos y se miran los mismos KPIs en cada mitad.
2. **Sin el merchant con más fraude** (`M46B66AE0E309595B`, 256 fraudes), que pesa mucho en los totales.

In [ ]:
rng = np.random.default_rng(0)
ids = np.array(sorted(set(merchant)))
half = dict(zip(ids, rng.integers(0, 2, len(ids))))
h = np.array([half[m] for m in merchant])


def chequeo(rows, label):
    out = {}
    d = PRE & ap & rows
    out["pre: fraude evitado (% del $)"] = amount[d & fr].sum() / amount[ap & fr & rows].sum()
    out["pre: TPV bueno rechazado (%)"] = amount[d & ~fr].sum() / amount[ap & ~fr & rows].sum()
    out["pre: equilibrio"] = amount[d & fr].sum() / amount[d & ~fr].sum()
    rem = ap & ~d & rows
    hv = (REVISION | RETENCION) & rem
    out["post: TPV retenido (%)"] = amount[hv].sum() / amount[rem].sum()
    out["post: fraude retenido (% del $)"] = amount[hv & fr].sum() / amount[rem & fr].sum()
    return pd.Series(out, name=label)


top = t.loc[t["is_fraud"] == 1, "merchant_id"].value_counts().index[0]
tab = pd.concat([
    chequeo(np.ones(N, bool), "todos"), chequeo(h == 0, "mitad A"), chequeo(h == 1, "mitad B"),
    chequeo(merchant != top, f"sin {top[:7]}"),
], axis=1)
tab.round(4).to_csv(DOCS / "chequeos.csv")
print(tab.round(4).to_string())

## Segmentos para los emisores (bonus)

Para mantener la reputación con los emisores, las transacciones aprobadas se agrupan en tres paquetes según el riesgo del merchant. El objetivo es que el paquete principal tenga una tasa de fraude baja y los paquetes de riesgo se envíen aparte.

- **Alto:** merchants de las reglas de rechazo, o con `risk_pct` de 0,99 o más.
- **Medio:** merchants de revisión o de retención automática, o con `risk_pct` entre 0,95 y 0,99.
- **Bajo:** el resto.

In [ ]:
alto = PRE | (risk >= 0.99)
medio = ~alto & (REVISION | RETENCION | (risk >= 0.95))
seg = np.where(alto, "alto", np.where(medio, "medio", "bajo"))
base = pd.DataFrame({"segmento": seg[ap], "fraude": fr[ap], "monto": amount[ap], "merchant": merchant[ap]})
seg_tab = base.groupby("segmento").agg(
    merchants=("merchant", "nunique"), transacciones=("fraude", "size"), tpv=("monto", "sum"), fraude=("fraude", "mean"),
).reindex(["bajo", "medio", "alto"])
seg_tab["% transacciones"] = seg_tab["transacciones"] / seg_tab["transacciones"].sum()
seg_tab["% tpv"] = seg_tab["tpv"] / seg_tab["tpv"].sum()
seg_tab.round(6).to_csv(DOCS / "segmentos.csv")
print(seg_tab.round({"fraude": 5, "% transacciones": 4, "% tpv": 4}).to_string())
print()
print(f"tasa de fraude de todo lo aprobado: {fr[ap].mean():.4%} | del paquete principal (bajo): {seg_tab.loc['bajo', 'fraude']:.4%}")

## Gráfico: lo que se gana y lo que se paga

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
colors = {"baja": "#378ADD", "media": "#D85A30", "alta": "#888780", "baja + media": "#534AB7"}
for name, g in cortes.groupby("volumen"):
    x = g["TPV bueno rechazado (% del bueno)"] * 100
    y = g["fraude evitado (% del $)"] * 100
    ax[0].plot(x, y, marker="o", label=f"volumen {name}", color=colors[name])
    for cut, xi, yi in zip(g["corte"], x, y):
        ax[0].annotate(f"{cut:.2f}", (xi, yi), fontsize=7, xytext=(4, 3), textcoords="offset points")
ax[0].plot(pre["TPV bueno rechazado (% del bueno)"] * 100, pre["fraude evitado (% del $)"] * 100, "k*", ms=16, label="regla elegida")
ax[0].plot([0, 35], [0, 35], color="#B4B2A9", ls="--", label="azar")
ax[0].set_xlabel("TPV bueno rechazado (% del TPV bueno)")
ax[0].set_ylabel("fraude evitado (% del $ de fraude)")
ax[0].set_title("Pre-autorización: rechazar (los números son el corte de risk_pct)")
ax[0].legend(fontsize=8)
labs, vals = ["bajo", "medio", "alto"], seg_tab["fraude"] * 100
ax[1].bar(labs, vals, color=["#378ADD", "#EF9F27", "#D85A30"])
ax[1].axhline(fr[ap].mean() * 100, color="#444441", ls="--", label="promedio de lo aprobado")
for i, v in enumerate(vals):
    ax[1].text(i, v + 0.003, f"{v:.3f}%", ha="center")
ax[1].set_ylabel("tasa de fraude (%)")
ax[1].set_title("Segmentos para los emisores")
ax[1].legend()
fig.tight_layout()
fig.savefig(DOCS / "reglas.png", dpi=110)
plt.close(fig)
print("guardado", DOCS)

## Trade-off de las reglas

Para cada regla, cuánto fraude se gana y cuánto TPV se paga, barriendo el corte de `risk_pct`.

- **Pre-autorización (izquierda):** eje x = TPV bueno rechazado; eje y = fraude evitado. La línea gris punteada es el azar (rechazar al azar evita la misma proporción que se rechaza). Las líneas rojas son el **equilibrio**: un punto por encima de la línea es rentable con ese margen de Clip sobre el TPV. Cuanto más empinada, más margen supone.
- **Post-autorización (derecha):** eje x = TPV retenido; eje y = fraude retenido. Retener no pierde TPV, solo lo demora, así que no hay equilibrio en dinero: el costo es la demora y la carga de revisión.

In [ ]:
cuts = np.round(np.arange(0.80, 0.9995, 0.01), 4)
cuts = np.unique(np.r_[cuts, 0.95, 0.99, 0.995, 0.999])


def pre_point(mask):
    d = mask & ap
    return (amount[d & ~fr].sum() / TPV_BUENO * 100, amount[d & fr].sum() / FRAUDE_D * 100)


def post_point(mask):
    h = mask & ap
    return (amount[h].sum() / TPV * 100, amount[h & fr].sum() / FRAUDE_D * 100)


fig, ax = plt.subplots(1, 2, figsize=(15, 5.8))

# ---- pre: rechazar
colors = {"baja": "#378ADD", "media": "#D85A30", "alta": "#888780"}
for vol_name in ("baja", "media", "alta"):
    pts = np.array([pre_point((risk >= c) & (volumen == vol_name)) for c in cuts])
    ax[0].plot(pts[:, 0], pts[:, 1], color=colors[vol_name], lw=1.6, label=f"volumen {vol_name}")
    for c in (0.90, 0.95, 0.99):
        i = int(np.argmin(np.abs(cuts - c)))
        ax[0].plot(*pts[i], "o", color=colors[vol_name], ms=5)
        ax[0].annotate(f"{c:.2f}", pts[i], fontsize=7, xytext=(4, -9), textcoords="offset points", color=colors[vol_name])
xmax = 14
xs = np.array([0, xmax])
ax[0].plot(xs, xs, color="#B4B2A9", ls="--", label="azar")
slope1 = MARGEN * TPV_BUENO / FRAUDE_D
ax[0].plot(xs, xs * slope1, color="#C0392B", ls=":", lw=1.5, label=f"equilibrio con margen {MARGEN:.0%}")
ax[0].plot(xs, xs * slope1 * 2, color="#C0392B", ls="-.", lw=1, label=f"equilibrio con margen {2 * MARGEN:.0%}")
ax[0].plot(pre["TPV bueno rechazado (% del bueno)"] * 100, pre["fraude evitado (% del $)"] * 100, "k*", ms=17, label="regla elegida")
ax[0].set_xlim(0, xmax)
ax[0].set_ylim(0, 30)
ax[0].set_xlabel("TPV bueno rechazado (% del TPV bueno)")
ax[0].set_ylabel("fraude evitado (% del $ de fraude)")
ax[0].set_title("Pre-autorización: rechazar (puntos = corte de risk_pct)")
ax[0].legend(fontsize=8, loc="lower right")

# ---- post: retener
for vol_name, color in (("alta", "#888780"), ("todos", "#534AB7")):
    sel = (volumen == "alta") if vol_name == "alta" else np.ones(N, bool)
    pts = np.array([post_point((risk >= c) & sel & ~PRE) for c in cuts])
    ax[1].plot(pts[:, 0], pts[:, 1], color=color, lw=1.6, label=f"retener por riesgo (volumen {vol_name})")
    for c in (0.90, 0.95, 0.99):
        i = int(np.argmin(np.abs(cuts - c)))
        ax[1].plot(*pts[i], "o", color=color, ms=5)
        ax[1].annotate(f"{c:.2f}", pts[i], fontsize=7, xytext=(4, -9), textcoords="offset points", color=color)
ax[1].plot([0, 40], [0, 40], color="#B4B2A9", ls="--", label="azar")
for k, name, marker in ((rev, "revisión manual", "s"), (aut, "retención automática", "^"), (both, "ambas", "*")):
    ax[1].plot(k["TPV retenido (%)"] * 100, k["fraude retenido (% del $)"] * 100, marker, color="#D85A30", ms=13 if marker == "*" else 9, label=name)
ax[1].set_xlim(0, 40)
ax[1].set_ylim(0, 60)
ax[1].set_xlabel("TPV retenido (% del TPV aprobado)")
ax[1].set_ylabel("fraude retenido (% del $ de fraude)")
ax[1].set_title("Post-autorización: retener el pago")
ax[1].legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(DOCS / "trade_off.png", dpi=110)
plt.close(fig)
print("guardado", DOCS / "trade_off.png")

## Retención por tipo de pago y por monto

El score es por merchant y no ve el fraude de los merchants chicos que venden sin tarjeta presente (CNP). Estas reglas miran la **transacción**: el tipo de pago y el monto. Los dos datos están disponibles al autorizar.

- **Rechazar por tipo de pago o monto no conviene.** El fraude se concentra en los montos altos, pero rechazar esas transacciones cuesta mucho TPV bueno: el equilibrio queda por debajo del margen (primera tabla). Se muestra para dejar el resultado negativo documentado.
- **Retener sí** (segunda tabla): no se pierde TPV, solo se demora. Se retiene lo aprobado que cumple alguna de dos condiciones.
  - **CNP con monto de 500 o más.**
  - **Monto de 20.000 o más**, de cualquier tipo.

Se miden sobre lo que queda aprobado después de las reglas de rechazo, y se compara el aporte sobre lo que ya retienen la revisión manual y la retención automática.

**Límites.** Los cortes (500 y 20.000) se eligieron mirando estas mismas transacciones. Las transacciones no tienen fecha, así que no se puede probar estabilidad en el tiempo. El fraude CNP está concentrado entre pocos merchants. Los chequeos de abajo separan por mitades de merchants y sacan al merchant con más fraude.

In [ ]:
pt = t["payment_type"].to_numpy()
CNP_MONTO = 500
ALTO_MONTO = 20_000

CNP = (pt == "CNP") & (amount >= CNP_MONTO)
MONTO_ALTO = amount >= ALTO_MONTO
ya_retenido = REVISION | RETENCION

# Rechazar por tipo de pago o monto: resultado negativo, el equilibrio no llega al margen.
rechazo = pd.DataFrame([
    pre_kpis(MONTO_ALTO, "Rechazar monto >= 20.000"),
    pre_kpis(amount >= 5000, "Rechazar monto >= 5.000"),
    pre_kpis(pt == "CNP", "Rechazar todo CNP"),
    pre_kpis(CNP, "Rechazar CNP con monto >= 500"),
])[["regla", "fraude evitado (% del $)", "TPV bueno rechazado (% del bueno)", "equilibrio"]]
print(rechazo.round(4).to_string(index=False))
print()

# Retener: no pierde TPV, solo lo demora.
actual = post_kpis(ya_retenido, remaining, "Actual: revisión + retención automática")
nuevas = [
    post_kpis(CNP, remaining, "Nueva: CNP con monto >= 500"),
    post_kpis(MONTO_ALTO, remaining, "Nueva: monto >= 20.000"),
    post_kpis(CNP | MONTO_ALTO, remaining, "Nuevas: ambas"),
]
todas = post_kpis(ya_retenido | CNP | MONTO_ALTO, remaining, "Actual + nuevas")
extra = (CNP | MONTO_ALTO) & remaining & ~ya_retenido
extra_tpv = amount[extra].sum() / amount[remaining].sum()
extra_fraude = amount[extra & fr].sum() / amount[remaining & fr].sum()
retencion = pd.DataFrame([actual] + nuevas + [todas]).round(5)
retencion["transacciones"] = [int((m & remaining).sum()) for m in (ya_retenido, CNP, MONTO_ALTO, CNP | MONTO_ALTO, ya_retenido | CNP | MONTO_ALTO)]
retencion.to_csv(DOCS / "retencion_cnp_monto.csv", index=False)
rechazo.round(5).to_csv(DOCS / "rechazo_cnp_monto.csv", index=False)
print(retencion.drop(columns="merchants").to_string(index=False))
print()
print(f"aporte de las nuevas sobre lo ya retenido: +{extra_tpv:.1%} del TPV, +{extra_fraude:.1%} del $ de fraude ({extra_fraude / extra_tpv:.2f} veces)")

In [ ]:
# Se sostiene: mitades de merchants y sin el merchant con más fraude.
rng = np.random.default_rng(0)
codigo, ids = pd.factorize(merchant)
mitad = np.isin(codigo, rng.choice(len(ids), len(ids) // 2, replace=False))
mayor = t.loc[ap & fr].groupby("merchant_id")["amount"].sum().idxmax()
subconjuntos = {"mitad 1": mitad, "mitad 2": ~mitad, f"sin {mayor[:8]}": merchant != mayor}


def parte(mask, sub):
    base = remaining & sub
    return amount[mask & base].sum() / amount[base].sum(), amount[mask & base & fr].sum() / amount[base & fr].sum()


filas = []
for nombre, sub in subconjuntos.items():
    for regla, mask in (("CNP con monto >= 500", CNP), ("monto >= 20.000", MONTO_ALTO), ("actual", ya_retenido), ("actual + nuevas", ya_retenido | CNP | MONTO_ALTO)):
        tpv_p, fraude_p = parte(mask, sub)
        filas.append({"subconjunto": nombre, "regla": regla, "TPV retenido (%)": tpv_p, "fraude retenido (% del $)": fraude_p})
chequeo = pd.DataFrame(filas).round(4)
print(chequeo.to_string(index=False))

cnp_fraude = t.loc[ap & fr & (pt == "CNP")].groupby("merchant_id")["amount"].sum()
print()
print(f"fraude CNP: {len(cnp_fraude)} merchants; los 10 mayores suman {cnp_fraude.nlargest(10).sum() / cnp_fraude.sum():.0%} del $, el mayor {cnp_fraude.max() / cnp_fraude.sum():.0%}")